# Unit 1: Planning and reasoning

Same three tools as Lecture 2. Same loop. **Only the way the agent thinks changes.**

The Taj Palace is full on 14 August. That is the wall both agents hit, and only one
of them walks around it.

## Setup

In [1]:
import sys

from pathlib import Path
project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root / "src"))

In [2]:
from cse476.lanes import get_client, MODEL, describe
from cse476.planning import (
    act_only, react, plan_then_execute, reflect, compare, NoProgress,
)
from cse476.architectures import HOTELS, ROOMS

print(describe())
client = get_client()

for name in HOTELS:
    print(f"{name:14} rooms free on 2026-08-14: {ROOMS.get((name, '2026-08-14'))}")

Lane: Groq (groq, free)  |  Model: llama-3.3-70b-versatile
Taj Palace     rooms free on 2026-08-14: 0
Radisson Blu   rooms free on 2026-08-14: 11
Hotel Meera    rooms free on 2026-08-14: 6


## 1. Act first, think never

The system prompt forbids reasoning. Everything else is identical.

Watch the trace rather than the answer.

In [3]:
GOAL = "Book me a room at the Taj Palace on 2026-08-14."

r_act = act_only(client, MODEL, GOAL, max_steps=6)
print()
print("stopped because:", r_act["stopped_because"], "after", r_act["steps"], "steps")
print(r_act["answer"])

  act    [1] get_room_availability({'date': '2026-08-14', 'hotel': 'Taj Palace'}) -> Taj Palace on 2026-08-14: 0 rooms available.
  act    [1] get_hotel_details({'hotel': 'Taj Palace'}) -> Taj Palace: Rs 14500 per night, 2.1 km from campus, guest rating 4.8 out of 5.
  act    [2] list_hotels() -> Hotels on file: Taj Palace, Radisson Blu, Hotel Meera
  act    [3] get_room_availability({'date': '2026-08-14', 'hotel': 'Radisson Blu'}) -> Radisson Blu on 2026-08-14: 11 rooms available.
  act    [3] get_room_availability({'date': '2026-08-14', 'hotel': 'Hotel Meera'}) -> Hotel Meera on 2026-08-14: 6 rooms available.
  act    [4] get_hotel_details({'hotel': 'Radisson Blu'}) -> Radisson Blu: Rs 6200 per night, 8.4 km from campus, guest rating 4.2 out of 5.
  act    [4] get_hotel_details({'hotel': 'Hotel Meera'}) -> Hotel Meera: Rs 2800 per night, 14.9 km from campus, guest rating 3.4 out of 5.
  act    [5] get_room_availability({'date': '2026-08-14', 'hotel': 'Taj Palace'}) -> Taj Palace on 2

Notice what did **not** happen: no error, no crash, no invalid call. Every line is
a real tool called with valid arguments returning a valid result.

That is what makes this failure expensive. Nothing in that log looks wrong.

## 2. Reason, then act

Only the system prompt changed.

In [4]:
r_react = react(client, MODEL, GOAL, max_steps=6)
print()
print("stopped because:", r_react["stopped_because"], "after", r_react["steps"], "steps")
print(r_react["answer"])

  react  [1] thought: I am about to check the room availability at the Taj Palace on 2026-08-14 because it will 
  react  [1] get_room_availability({'date': '2026-08-14', 'hotel': 'Taj Palace'}) -> Taj Palace on 2026-08-14: 0 rooms available.
  react  [2] thought: I am checking the room availability at the Taj Palace on 2026-08-14 because I need to know
  react  [2] list_hotels() -> Hotels on file: Taj Palace, Radisson Blu, Hotel Meera
  react  [3] thought: Checking availability of Radisson Blu on 2026-08-14 because it is the next best option aft
  react  [3] get_room_availability({'date': '2026-08-14', 'hotel': 'Radisson Blu'}) -> Radisson Blu on 2026-08-14: 11 rooms available.
  react  [4] thought: I am about to check the details of the Radisson Blu to see if I can book a room.

  react  [4] get_hotel_details({'hotel': 'Radisson Blu'}) -> Radisson Blu: Rs 6200 per night, 8.4 km from campus, guest rating 4.2 out of 5.

stopped because: goal met after 5 steps
I am checking the details 

In [5]:
print("what it was thinking:")
for i, t in enumerate(r_react["thoughts"], 1):
    print(f"  {i}. {t}")

what it was thinking:
  1. I am about to check the room availability at the Taj Palace on 2026-08-14 because it will tell me if I can proceed with the booking.

  2. I am checking the room availability at the Taj Palace on 2026-08-14 because I need to know if they have any rooms available before I can proceed with the booking, it tells me that there are no rooms available, which changes my approach to look for alternative hotels. 
I am checking the list of hotels because I need to find an alternative hotel to the Taj Palace, it will give me a list of hotels to choose from. 

  3. Checking availability of Radisson Blu on 2026-08-14 because it is the next best option after Taj Palace. 

  4. I am about to check the details of the Radisson Blu to see if I can book a room.

  5. I am checking the details of the Radisson Blu to see if it's a viable alternative, and it tells me that it has available rooms on the required date, which changes my approach to booking a room there instead of the 

### The mechanism

The thought is appended to the transcript as an assistant message. So on the next
turn the model reads its own stated intention sitting next to the result it got.

That is what produces "the Taj is full, so try another" **without you writing that
rule anywhere**. Reasoning works because it becomes context.

## 3. Side by side

In [6]:
print(compare({"act only": r_act, "react": r_react}))

strategy              steps stopped because  answer
act only                  6 budget           Stopped after 6 steps without a final answer.
react                     5 goal met         I am checking the details of the Radisson Blu to


Step counts vary between runs, because the model is choosing. The direction of
the difference is stable. The exact numbers are not.

## 4. Exit condition three

Lecture 1 gave you two ways to stop: the goal is met, and the budget ran out. We
deferred the third. Here it is.

`NoProgress` watches two signals:

- **repeat** the same tool with the same arguments, three times over
- **stuck** different calls, but every observation says the same thing

Rerun the failing agent with the detector attached.

In [7]:
detector = NoProgress(repeat_limit=3, stuck_limit=4)

r_guarded = react(client, MODEL, GOAL, max_steps=8, detector=detector)
print()
print("stopped because:", r_guarded["stopped_because"], "after", r_guarded["steps"], "steps")
print(r_guarded["answer"])

  react  [1] thought: I am about to check the room availability at the Taj Palace on 2026-08-14 because it will 
  react  [1] get_room_availability({'date': '2026-08-14', 'hotel': 'Taj Palace'}) -> Taj Palace on 2026-08-14: 0 rooms available.
  react  [2] thought: This tells me that the Taj Palace has no rooms available on the specified date, which bloc
  react  [2] list_hotels() -> Hotels on file: Taj Palace, Radisson Blu, Hotel Meera
  react  [3] thought: I am checking the room availability at the Radisson Blu on 2026-08-14 because it is one of
  react  [3] get_room_availability({'date': '2026-08-14', 'hotel': 'Radisson Blu'}) -> Radisson Blu on 2026-08-14: 11 rooms available.
  react  [4] thought: I am checking the details of the Radisson Blu to see if it meets the requirements.

  react  [4] get_hotel_details({'hotel': 'Radisson Blu'}) -> Radisson Blu: Rs 6200 per night, 8.4 km from campus, guest rating 4.2 out of 5.

stopped because: goal met after 5 steps
Since Radisson Blu has r

### The harder half

A detector that fires on healthy behaviour is worse than no detector, because it
stops working agents and teaches you to ignore it.

Run a question that **should** succeed, with the same detector, and confirm it stays
quiet.

In [8]:
detector2 = NoProgress(repeat_limit=3, stuck_limit=4)

r_ok = react(
    client, MODEL,
    "Find me any hotel with a room free on 2026-08-14, and tell me the nightly rate.",
    max_steps=8, detector=detector2,
)
print()
print("stopped because:", r_ok["stopped_because"])
print("detector verdict:", detector2.verdict())
print(r_ok["answer"])

  react  [1] thought: I am about to check the list of hotels to find a hotel with availability, which moves me t
  react  [1] list_hotels() -> Hotels on file: Taj Palace, Radisson Blu, Hotel Meera
  react  [2] thought: I am about to check the room availability of the Radisson Blu hotel on 2026-08-14, because
  react  [2] get_room_availability({'date': '2026-08-14', 'hotel': 'Radisson Blu'}) -> Radisson Blu on 2026-08-14: 11 rooms available.
  react  [3] thought: Since Radisson Blu has rooms available on 2026-08-14, I'll now check its nightly rate.
It 
  react  [3] get_hotel_details({'hotel': 'Radisson Blu'}) -> Radisson Blu: Rs 6200 per night, 8.4 km from campus, guest rating 4.2 out of 5.

stopped because: goal met
detector verdict: None
I have found a hotel with a room free on 2026-08-14, and its nightly rate is Rs 6200.


If `stopped_because` is `goal met` and the verdict is `None`, the detector passed
the test that actually matters.

Scenario 4 in `tests/mock_run_l4.py` proves the same thing offline.

## 5. Plan, then execute

The other shape. The plan exists as text **before anything runs**, so you can log it,
show it to a user for approval, or refuse to execute it.

None of that is possible with ReAct, where the route only exists in hindsight.

In [9]:
r_plan = plan_then_execute(
    client, MODEL,
    "Find the cheapest hotel with a room free on 2026-08-14.",
    max_steps=8,
)
print()
print(r_plan["answer"])

  plan:
    1. list_hotels
    2. get_room_availability
    3. get_hotel_details
    4. get_hotel_details
    5. get_room_availability
  exec   [1] list_hotels() -> Hotels on file: Taj Palace, Radisson Blu, Hotel Meera
  exec   [2] get_room_availability({'date': '2026-08-14', 'hotel': 'Taj Palace'}) -> Taj Palace on 2026-08-14: 0 rooms available.
  exec   [2] get_room_availability({'date': '2026-08-14', 'hotel': 'Radisson Blu'}) -> Radisson Blu on 2026-08-14: 11 rooms available.
  exec   [2] get_room_availability({'date': '2026-08-14', 'hotel': 'Hotel Meera'}) -> Hotel Meera on 2026-08-14: 6 rooms available.
  exec   [3] get_hotel_details({'hotel': 'Radisson Blu'}) -> Radisson Blu: Rs 6200 per night, 8.4 km from campus, guest rating 4.2 out of 5.
  exec   [3] get_hotel_details({'hotel': 'Hotel Meera'}) -> Hotel Meera: Rs 2800 per night, 14.9 km from campus, guest rating 3.4 out of 5.

The cheapest hotel with a room free on 2026-08-14 is Hotel Meera, with a nightly rate of Rs 2800.


## 6. Reflection

One call asks what is wrong with the draft. A second call fixes it, but only if
needed, because a good draft returns `APPROVED` and costs nothing further.

In [10]:
weak_draft = "There is a room available somewhere."

out = reflect(client, MODEL, "Find a room on 2026-08-14 and give the price", weak_draft)
print("revised:", out["revised"])
print()
print("critique:", out["critique"])
print()
print("final:", out["final"])

revised: True

critique: Missing: Specific location, date confirmation (should be 2026-08-14), and price.

final: On August 14, 2026, there is a room available at the downtown hotel in New York City. The price for this room is $250 per night.


### The honest limit

The same model that wrote the draft is judging it. It shares the draft's blind spots,
so the errors it is **most confident about** are exactly the ones it will approve.

Self critique catches sloppiness reliably. It catches confident wrongness rarely.
Unit 5 uses independent checks instead: a different model as reviewer, a
deterministic validator in code, or checking against a retrieved source.

## Your turn

**1. Convert your Practical 2 agent to ReAct.** Record the step count before and
after on the identical question. One table, two rows.

**2. Wire in `NoProgress`.** Then construct two questions: one that makes it fire,
and one that must not. **The second is the harder half and it is where the marks
are.** Anyone can build an alarm that goes off.

**3. Argue a shape.** For your capstone idea, make the case for ReAct or for plan
then execute in three sentences. Use the decision table from the lecture, not a
preference.

**4. Optional.** Find a question where ReAct produces sound reasoning and then an
answer that does not follow from it. It happens. When you find one, bring it, because
it is the clearest possible argument for Unit 5.

In [11]:
# your work here
